In [ ]:
import pandas as pd

In [ ]:
class ColumnGenerator:
    pass

In [ ]:
def process_df( df, column_generator:list=[] ):
    for col_name, columns, func in column_generator:
        df[col_name] = func( df[columns]  ) # columns is a list of string
   
   
    # Price returns and differences
    df['return_open_close'] = (df['close'] - df['open']) / df['open']
    df['return_prev_close'] = df['close'].pct_change()
    df['rolling_return_3'] = df['return_prev_close'].rolling(window=3).mean()
    df['rolling_return_5'] = df['return_prev_close'].rolling(window=5).mean()

    # Volatility/ATR
    high, low, close = df['high'], df['low'], df['close']
    prev_close = close.shift(1)
    tr = pd.concat([
        (high - low),
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis=1).max(axis=1)

    # Volumne Features
    df['volume_pct_change'] = df['volume'].pct_change()
    df['volume_rolling_mean_5'] = df['volume'].rolling(window=5).mean()
    df['volume_rolling_mean_10'] = df['volume'].rolling(window=10).mean()
    
    
    df['atr_14'] = tr.rolling(window=14, min_periods=1).mean()
    df['rolling_std_5'] = df['return_prev_close'].rolling(window=5).std()
    df['rolling_std_10'] = df['return_prev_close'].rolling(window=10).std()


    


    df = df.reset_index(drop=True)
    return df

In [ ]:
import glob

all_files = glob.glob('simulation_results/*.csv')
dfs = []

for file in all_files:
    df = pd.read_csv(file)
    # Optionally add identifiers if needed
    # df['date'] = extract_date_from_filename(file)
    # df['symbol'] = extract_symbol_from_filename(file)
    df = process_df(df)
    dfs.append(df)

full_data = pd.concat(dfs, ignore_index=True)


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Assuming full_data has all feature-engineered columns and 'action_quality' and 'time'

# 1. Encode target variable
le = LabelEncoder()
full_data['action_quality_enc'] = le.fit_transform(full_data['action_quality'])

# 2. Sort data by time (ascending)
full_data = full_data.sort_values('time').reset_index(drop=True)

# 3. Separate features and target
X = full_data.drop(columns=['action_quality', 'action_quality_enc', 'time'])
y = full_data['action_quality_enc']

# 4. Split into train and test sets (80/20 by time)
split_idx = int(len(full_data) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 5. Compute class weights for imbalance handling
classes = list(le.transform(le.classes_))
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# 6. (Optional) Feature scaling for models that need it (skip if using tree-based models)
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

# 7. Train Random Forest classifier with class weights
clf = RandomForestClassifier(class_weight=class_weight_dict, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# 8. Make predictions and evaluate
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
# scaler is already fitted from training phase

X_new_scaled = scaler.transform([X_new])  # Scale new data point using training params
prediction = model.predict(X_new_scaled)


In [ ]:
import joblib

# After training
joblib.dump(model, 'decider_model.pkl')
joblib.dump(scaler, 'scaler.pkl')


In [ ]:
model = joblib.load('decider_model.pkl')
scaler = joblib.load('scaler.pkl')

# Prepare new data features (X_new) same way as training
X_new_scaled = scaler.transform([X_new])
prediction = model.predict(X_new_scaled)
